# Task 3: Soft Mixture-of-Experts Validation & Routing Analysis

Compares the soft MoE model against the Task 1 universal DAE and Task 2b oracle
specialists.  Also visualises expert routing weights and blended predictions as
required by Plan 6 §3.6.


In [ ]:
# 1. Imports
import sys
from pathlib import Path
from collections import defaultdict
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))

from src.device_utils import get_device
from src.models_dae import ConvDAE
from src.models_classifier import CorruptionClassifier, SoftMoERestorer
from src.checkpoint_utils import load_checkpoint
from src.evaluation import PetBenchmarkDataset, image_quality_metrics

DEVICE = get_device()
CLASS_NAMES  = ["clean", "salt", "blur", "occlusion"]
BRANCH_NAMES = ["identity", "salt", "blur", "occlusion"]


In [ ]:
# 2. Checkpoint paths
MOE_CKPT    = RESEARCH_ROOT / "checkpoints" / "task3_soft_moe_best.pt"
TASK1_CKPT  = RESEARCH_ROOT / "checkpoints" / "task1_universal_ae_best.pt"
SPECIALIST_CKPTS = {
    "salt":     RESEARCH_ROOT / "checkpoints" / "task2b_specialist_salt_best.pt",
    "blur":     RESEARCH_ROOT / "checkpoints" / "task2b_specialist_blur_best.pt",
    "occlusion":RESEARCH_ROOT / "checkpoints" / "task2b_specialist_occlusion_best.pt",
}


## 1. Load Trained Soft MoE and Comparison Models

In [ ]:
# 3. Load models
required = [MOE_CKPT, TASK1_CKPT, *SPECIALIST_CKPTS.values()]
missing  = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Complete Tasks 1-3 training first. Missing: " + ", ".join(missing))

def load_dae_from_ckpt(path):
    state  = torch.load(path, map_location=DEVICE, weights_only=False)["model_state_dict"]
    base_c = state["enc1.0.weight"].shape[0]
    latent = state["fc_z.weight"].shape[0]
    m = ConvDAE(base_channels=base_c, latent_dim=latent).to(DEVICE)
    m.load_state_dict(state)
    m.eval()
    return m

# Task 1 universal model
task1 = load_dae_from_ckpt(TASK1_CKPT)

# Oracle specialists
specialists = {kind: load_dae_from_ckpt(path) for kind, path in SPECIALIST_CKPTS.items()}

# Soft MoE
moe_saved  = torch.load(MOE_CKPT, map_location=DEVICE, weights_only=False)
moe_state  = moe_saved["model_state_dict"]
gate_base  = moe_state["gate.conv1.0.weight"].shape[0]
gate = CorruptionClassifier(base_channels=gate_base).to(DEVICE)
salt_m  = ConvDAE(base_channels=moe_state["specialist_salt.enc1.0.weight"].shape[0],
                  latent_dim=moe_state["specialist_salt.fc_z.weight"].shape[0]).to(DEVICE)
blur_m  = ConvDAE(base_channels=moe_state["specialist_blur.enc1.0.weight"].shape[0],
                  latent_dim=moe_state["specialist_blur.fc_z.weight"].shape[0]).to(DEVICE)
occl_m  = ConvDAE(base_channels=moe_state["specialist_occlusion.enc1.0.weight"].shape[0],
                  latent_dim=moe_state["specialist_occlusion.fc_z.weight"].shape[0]).to(DEVICE)
moe = SoftMoERestorer(gate, salt_m, blur_m, occl_m).to(DEVICE)
load_checkpoint(MOE_CKPT, moe, device=DEVICE)
moe.eval()
print(f"MoE loaded from epoch {moe_saved.get('epoch')} – val_loss={moe_saved.get('val_loss'):.4f}")


## 2. Quantitative Comparison: Universal DAE vs Oracle Specialist vs Soft MoE

In [ ]:
# 4. Benchmark evaluation
benchmark = PetBenchmarkDataset(RESEARCH_ROOT / "test_manifest_official.json")
loader    = DataLoader(benchmark, batch_size=16, shuffle=False, num_workers=2)

summary       = defaultdict(lambda: defaultdict(float))
counts        = defaultdict(int)
routing_by_cls = defaultdict(list)   # (kind, severity) -> list of weight arrays
blended        = []                  # examples where 2+ branches carry >= 0.2

with torch.inference_mode():
    for corrupted, clean, labels, kinds, severities, names in loader:
        corrupted, clean = corrupted.to(DEVICE), clean.to(DEVICE)

        universal            = task1(corrupted).clamp(0, 1)
        mixture, weights, _  = moe(corrupted)
        mixture              = mixture.clamp(0, 1)

        for i, kind in enumerate(kinds):
            sev = severities[i]
            key = (kind, sev)
            counts[key] += 1

            if kind == "clean":
                oracle_out = corrupted[i:i+1]
            else:
                oracle_out = specialists[kind](corrupted[i:i+1]).clamp(0, 1)

            for model_name, out in [
                ("universal",        universal[i:i+1]),
                ("oracle_specialist",oracle_out),
                ("soft_moe",         mixture[i:i+1]),
            ]:
                m = image_quality_metrics(out, clean[i:i+1])
                for metric, val in m.items():
                    summary[(model_name, *key)][metric] += val

            routing_by_cls[key].append(weights[i].cpu().numpy())

            # Track examples with distributed routing
            sorted_w = weights[i].sort(descending=True).values
            if sorted_w[1] >= 0.2 and len(blended) < 8:
                blended.append({
                    "input":   corrupted[i].cpu(),
                    "target":  clean[i].cpu(),
                    "output":  mixture[i].cpu(),
                    "weights": weights[i].cpu().numpy(),
                    "kind": kind, "name": names[i],
                })

# Build comparison DataFrame
rows = [
    {"model": model_name, "corruption": kind, "severity": severity,
     **{m: v / counts[(kind, severity)] for m, v in metrics.items()}}
    for (model_name, kind, severity), metrics in sorted(summary.items())
]
comparison_df = pd.DataFrame(rows)
display(comparison_df)


## 3. Expert Routing Weight Heatmap (Plan 6 §3.6)

In [ ]:
# 5. Routing heatmap
sorted_keys = sorted(routing_by_cls.keys())
heat   = np.array([np.stack(routing_by_cls[k]).mean(0) for k in sorted_keys])
labels = [f"{k[0]} / {k[1]}" for k in sorted_keys]

plt.figure(figsize=(8, max(4, len(labels) * 0.38)))
sns.heatmap(
    heat, annot=True, fmt=".2f", cmap="YlGnBu",
    vmin=0, vmax=1, xticklabels=BRANCH_NAMES, yticklabels=labels
)
plt.xlabel("Branch")
plt.ylabel("True corruption / severity")
plt.title("Average Expert Routing Weights per Input Condition")
plt.tight_layout()
plt.show()

overall = heat.mean(0)
print("Overall branch utilisation:",
      dict(zip(BRANCH_NAMES, overall.round(3))))
inactive = [b for b, u in zip(BRANCH_NAMES, overall) if u < 0.05]
dominant = [b for b, u in zip(BRANCH_NAMES, overall) if u > 0.7]
if inactive:  print("Potentially inactive branches:", inactive)
if dominant:  print("Potentially dominant branches:", dominant)


## 4. Multi-Expert Blending Visualizations

In [ ]:
# 6. Examples where 2+ branches are active
if not blended:
    print("No sample had two branches with weight >= 0.2 – try a lower threshold.")
else:
    fig, axes = plt.subplots(len(blended), 3,
                             figsize=(10, 2.8 * len(blended)), squeeze=False)
    for row, record in zip(axes, blended):
        for ax, image, title in zip(
            row,
            [record["input"], record["output"], record["target"]],
            ["Input", "Soft mixture", "Target"]
        ):
            ax.imshow(image.permute(1, 2, 0).numpy().clip(0, 1))
            ax.set_title(title, fontsize=9)
            ax.axis("off")
        w_str = ", ".join(
            f"{name}={w:.2f}" for name, w in zip(BRANCH_NAMES, record["weights"])
        )
        print(f"{record['name']} [{record['kind']}]: {w_str}")
    plt.tight_layout()
    plt.show()


## 5. Dominant vs Distributed Routing Examples

In [ ]:
# 7. Find examples where one branch dominates (weight > 0.8) vs distributed
with torch.inference_mode():
    sample_batch_corr, sample_batch_clean, _, sample_kinds, sample_sevs, sample_names = next(iter(loader))
    sample_batch_corr = sample_batch_corr.to(DEVICE)
    _, sample_weights, _ = moe(sample_batch_corr)

dominant_idx   = sample_weights.max(1).values.argmax().item()
distributed_idx = (sample_weights.max(1).values - sample_weights.min(1).values).argmin().item()

fig, axes = plt.subplots(2, 3, figsize=(11, 5))
for row_i, idx, title_prefix in [
    (0, dominant_idx,    "Dominant routing"),
    (1, distributed_idx, "Distributed routing"),
]:
    img_in  = sample_batch_corr[idx].cpu()
    img_gt  = sample_batch_clean[idx]
    img_out = moe(img_in.unsqueeze(0).to(DEVICE))[0].squeeze(0).cpu()
    w       = sample_weights[idx].cpu().numpy()
    for col, (image, title) in enumerate(zip(
        [img_in, img_out, img_gt],
        ["Input", "MoE output", "Target"]
    )):
        axes[row_i, col].imshow(image.permute(1, 2, 0).numpy().clip(0, 1))
        axes[row_i, col].set_title(
            f"{title_prefix}\n{title} | " +
            ", ".join(f"{n}={v:.2f}" for n, v in zip(BRANCH_NAMES, w))
            if col == 0 else title, fontsize=7
        )
        axes[row_i, col].axis("off")
plt.tight_layout()
plt.show()
